In [1]:
# =============================================================================
# Antibiotic Resistance Prediction using Proximity, Frequency, and G-score
# =============================================================================
# 
# This notebook trains and evaluates Random Forest classifiers on mutation-level
# features (mutation frequency, 3D structural proximity to known resistance-conferring
# mutations, and g-scores) to predict antibiotic resistance in Mycobacterium tuberculosis.
# It includes baseline modeling,combined modeling and predictions for uncertain mutations (Category 3).

In [47]:
## load dependencies
import pandas as pd
import numpy as np
import re
from evcouplings.compare import DistanceMap
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score,roc_auc_score, roc_curve, auc, ConfusionMatrixDisplay)
from scipy import stats
import os,json
from utils import *

## data preprocessing

In [39]:
# ----------------------------
# STEP 1: Load and Inspect Dataset
# ----------------------------
# new_data: sanjana's data
final_df = pd.read_csv("../data/new_data/all_proteins_freq_proximity_gscore.csv")
print("Unique genes:", np.unique(final_df['gene']))
print("Data shape:", final_df.shape)
final_df['one_letter_mutation'] = final_df['one_letter_mutation'].str.replace("p.", "").str.strip()


Unique genes: ['Rv0678' 'Rv1258c' 'Rv1979c' 'Rv2983' 'ahpC' 'atpE' 'clpC1' 'ddn' 'eis'
 'embA' 'embB' 'embC' 'ethA' 'fbiA' 'fbiB' 'fbiC' 'fgd1' 'gid' 'gyrA'
 'gyrB' 'inhA' 'katG' 'mmpL5' 'mmpS5' 'mshA' 'panD' 'pepQ' 'pncA' 'rplC'
 'rpoB' 'rpsL' 'tlyA' 'ubiA' 'whiB7']
Data shape: (9365, 15)


In [ ]:
# STEP 2: Basic Preprocessing
# ----------------------------
# Separate Category 3 entries (uncertain mutations)
category_3_data = final_df[final_df['confidence'] == "3) Uncertain significance"]
print(category_3_data.shape)

# Retain only labeled data (non-Category 3) for supervised learning
label_data = final_df[final_df['confidence'] != "3) Uncertain significance"]

# Target conversion to binary
label_data['binary_confidence'] = label_data['confidence'].map({
    "1) Assoc w R": 1,
    "2) Assoc w R - Interim": 1,
    "4) Not assoc w R - Interim": 0,
    "5) Not assoc w R": 0
})

# Drop missing targets
label_data = label_data[label_data['binary_confidence'].notna()]

#some debugging steps
print(label_data['confidence'].value_counts())
print(np.unique(label_data['confidence']))

# Identify rows where the 'frequency' column is NaN
nan_frequency_rows = label_data[label_data['frequency'].isna()]

diverse_sample_mutation_table = label_data.groupby('frequency').first().reset_index()
diverse_sample_mutation_table.head()

(8697, 15)
confidence
1) Assoc w R                  397
2) Assoc w R - Interim        192
4) Not assoc w R - Interim     47
5) Not assoc w R               32
Name: count, dtype: int64
['1) Assoc w R' '2) Assoc w R - Interim' '4) Not assoc w R - Interim'
 '5) Not assoc w R']


/var/folders/90/7pfcyv8s59v1d678dnl3bgxw0000gq/T/ipykernel_39311/396068430.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label_data['binary_confidence'] = label_data['confidence'].map({


,frequency,drug,gene,variant,confidence,one_letter_mutation,phenotype,position,Proximity_1D,Nearest_1D_Index,Proximity_to_R_Conferring,Nearest_Mutation_Index,G_score,WHO_Adjusted_Position,Proximity_to_R_Conferring_zeroed,binary_confidence
0,0.000000,Clofazimine,Rv0678,Rv0678_p.Met146Thr,2) Assoc w R - Interim,M146T,Resistant,146,4.0,142.0,2.870637,142.0,-0.455283,22.0,2.870637,1
1,0.004854,Ethambutol,embB,embB_p.Arg213Gln,5) Not assoc w R,R213Q,Susceptible,213,83.0,296.0,7.415558,497.0,1.143737,NaN,7.415558,0
2,0.007299,Ethambutol,embA,embA_p.Ala201Thr,4) Not assoc w R - Interim,A201T,Susceptible,201,407.0,608.0,64.663760,608.0,1.797703,NaN,64.663760,0
3,0.009091,Clofazimine,Rv1979c,Rv1979c_p.Asp286Gly,4) Not assoc w R - Interim,D286G,Susceptible,286,123.0,409.0,47.945507,409.0,-0.146128,NaN,47.945507,0
4,0.009569,Moxifloxacin,gyrB,gyrB_p.Pro94Leu,4) Not assoc w R - Interim,P94L,Susceptible,94,352.0,446.0,54.035292,447.0,-1.423563,NaN,54.035292,0


## run the baseline and combined models

In [41]:
feature_sets = {
    # "Frequency": ['frequency'],
    "Proximity": ['Proximity_to_R_Conferring'],
    "Proximity_zeroed": ["Proximity_to_R_Conferring_zeroed"],
    "Proximity_1d":["Proximity_1D"],
    "G-score": ['G_score'],
    # "Frequency + Proximity": ['frequency', 'Proximity_to_R_Conferring'],
    # "Frequency + G-score": ['frequency', 'G_score'],
    # "Combined": ['frequency', 'Proximity_to_R_Conferring', 'G_score']
}


In [57]:
results = {}

def model_training(X_train, y_train,X_test):
    rf = LogisticRegression(class_weight="balanced")
    rf.fit(X_train, y_train)
    y_pred_test = rf.predict(X_test)
    y_prob_test = rf.predict_proba(X_test)
    return rf, y_pred_test, y_prob_test

for label, numeric_columns in feature_sets.items():
    print(f"\n=== Running model for: {label} ===")

    # Step 3: Feature selection
    label_data[numeric_columns] = label_data[numeric_columns].fillna(0)
    
    # Step 4: Split
    X_train, X_test, y_train, y_test, train_indices, test_indices = data_split(label_data, numeric_columns)
    
    # Step 5: Train and evaluate
    trained_rf, y_pred_test, y_prob_test = model_training(X_train, y_train, X_test)
    print("Confusion Matrix:")
    cm = confusion_matrix(y_test, y_pred_test, labels=[0,1]) #0-resistant
    print("Confusion matrix", cm)
    tn,fp,fn, tp = cm.ravel()
    sensitivity = tp / (tp+fn) #sensitivity resistant
    specificity = tn / (tn+fp) # specificity resistant
    print(f" Sensitivity: {sensitivity}, specificity: {specificity}")
    accuracy, roc_auc, precision = performance_evaluation(y_test, y_pred_test, y_prob_test)
    print("Accuracy:", accuracy)
    print("AUC:", roc_auc)
    print("Precision:", precision)

    # Step 6: Feature importance
    # feature_importances = plot_feature_importance(trained_rf, numeric_columns)
    feature_importances = trained_rf.coef_

    # Step 7: Category 3 prediction
    (
        misclassified_original_indices,
        misclassified_samples,
        confidence_0_count,
        confidence_0_percentage,
        confidence_1_count,
        confidence_1_percentage
    ) = predict_category_3(
        category_3_data,
        trained_rf,
        final_df,
        numeric_columns,
        y_test,
        y_pred_test,
        test_indices
    )

    # Store all results
    results[label] = {
        "features": numeric_columns,
        "accuracy": accuracy,
        "precision": precision,
        "roc_auc": roc_auc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "feature_importance": feature_importances,
        "misclassified_indices": misclassified_original_indices,
        "misclassified_samples": misclassified_samples,
        "category_3_predictions": {
            "resistant_count": confidence_0_count,
            "susceptible_count": confidence_1_count,
            "resistant_percentage": confidence_0_percentage,
            "susceptible_percentage": confidence_1_percentage
        }
    }



=== Running model for: Proximity ===
Confusion Matrix:
Confusion matrix [[ 13  11]
 [ 12 165]]
 Sensitivity: 0.9322033898305084, specificity: 0.5416666666666666
Classification Report:
              precision    recall  f1-score   support

           0       0.52      0.54      0.53        24
           1       0.94      0.93      0.93       177

    accuracy                           0.89       201
   macro avg       0.73      0.74      0.73       201
weighted avg       0.89      0.89      0.89       201

Accuracy: 0.8855721393034826
AUC: {0: 0.6839689265536724, 1: 0.6839689265536724}
Precision: 0.8876492537313432

=== Running model for: Proximity_zeroed ===
Confusion Matrix:
Confusion matrix [[ 12  12]
 [ 14 163]]
 Sensitivity: 0.9209039548022598, specificity: 0.5
Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.50      0.48        24
           1       0.93      0.92      0.93       177

    accuracy                        

/Users/annagreen/Desktop/CICS_local/Green_mutation_clustering_local/codebase/MTB_Mut_Clust/scripts/utils.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  category_3_data[numeric_columns] = category_3_data[numeric_columns].fillna(0)
/Users/annagreen/anaconda3/envs/info390/lib/python3.12/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/Users/annagreen/Desktop/CICS_local/Green_mutation_clustering_local/codebase/MTB_Mut_Clust/scripts/utils.py:80: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.or

In [51]:
def find_invalid_keys(obj, path="root"):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if not isinstance(k, (str, int, float, bool, type(None))):
                print(f"[Invalid Key] Path: {path} → Key: {k} (type: {type(k).__name__})")
            find_invalid_keys(v, path=f"{path}['{k}']")
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            find_invalid_keys(item, path=f"{path}[{i}]")


# Check before saving
find_invalid_keys(results)

def convert_keys_to_str(obj):
    if isinstance(obj, dict):
        return {str(k): convert_keys_to_str(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_keys_to_str(i) for i in obj]
    else:
        return obj
cleaned_results = convert_keys_to_str(results)

[Invalid Key] Path: root['Proximity']['roc_auc'] → Key: 0 (type: int64)
[Invalid Key] Path: root['Proximity']['roc_auc'] → Key: 1 (type: int64)
[Invalid Key] Path: root['Proximity_zeroed']['roc_auc'] → Key: 0 (type: int64)
[Invalid Key] Path: root['Proximity_zeroed']['roc_auc'] → Key: 1 (type: int64)
[Invalid Key] Path: root['Proximity_1d']['roc_auc'] → Key: 0 (type: int64)
[Invalid Key] Path: root['Proximity_1d']['roc_auc'] → Key: 1 (type: int64)
[Invalid Key] Path: root['G-score']['roc_auc'] → Key: 0 (type: int64)
[Invalid Key] Path: root['G-score']['roc_auc'] → Key: 1 (type: int64)


In [52]:
with open("../results/sanjana_baseline_combined_results.json", "w") as f:
    for label in cleaned_results:
        for key in cleaned_results[label]:
            if not isinstance(key, (str, int, float, bool, type(None))):
                print("Invalid key type:", key, type(key))
    json.dump(cleaned_results, f, indent=4, default=str)

print("\nAll model combinations evaluated and saved.")


All model combinations evaluated and saved.


In [55]:
def generate_markdown_table(results_dict):
    table_md = "| Model | AUC (avg) | Precision | Sensitivity | Specificity\n"
    table_md += "|--------|------------|-----------|-----------|-----------|\n"

    for model_name, metrics in results_dict.items():

        prec = f"{metrics.get('precision', 0) * 100:.1f}%"

        roc_auc = metrics.get('roc_auc', None)
        sensitivity = f"{metrics.get('sensitivity', 0) * 100:.1f}%"
        specificity = f"{metrics.get('specificity', 0) * 100:.1f}%"
        if isinstance(roc_auc, dict):
            # Average over classes
            auc_val = np.mean(list(roc_auc.values()))
        else:
            auc_val = roc_auc

        auc_str = f"{auc_val * 100:.1f}%" if auc_val is not None else "N/A"

        table_md += f"| {model_name} | {auc_str} | {prec} | {sensitivity} | {specificity} |\n"

    return table_md


In [56]:
markdown_output = generate_markdown_table(cleaned_results)
print(markdown_output)

| Model | AUC (avg) | Precision | Sensitivity | Specificity
|--------|------------|-----------|-----------|-----------|
| Proximity | 68.4% | 88.8% | 93.2% | 54.2% |
| Proximity_zeroed | 75.2% | 87.5% | 92.1% | 50.0% |
| Proximity_1d | 63.5% | 84.8% | 94.4% | 29.2% |
| G-score | 74.0% | 87.1% | 64.4% | 79.2% |



## evaluation on new data

In [11]:
import re

def normalize_variant(variant):
    """Converts rpoB_p.Ser450Leu → rpoB_S450L (1-letter style)"""
    if pd.isna(variant):
        return None
    match = re.match(r'(\w+)_p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})', variant)
    if match:
        gene, wt, pos, mut = match.groups()
        wt1 = wt[0]
        mut1 = mut[0]
        return f"{gene}_{wt1}{pos}{mut1}"
    return variant  # fallback for non-matching strings


In [17]:
import pandas as pd

# Load the 2021 and 2023 datasets
file_2021 = "../data/new_data/all_proteins_freq_proximity_gscore.csv"
df_2021 = pd.read_csv(file_2021)
df_2023 = final_df

In [18]:
# Normalize 2023
df_2023["variant"] = df_2023["variant"].apply(normalize_variant)

# Optional: verify alignment visually
# display(df_2021[["variant", "normalized_variant", "one_letter_mutation", "normalized_mutation"]].head())
# display(df_2023[["variant", "normalized_variant", "one_letter_mutation", "normalized_mutation"]].head())


In [19]:
merge_cols = ["gene", "variant", "one_letter_mutation"]

merged_df = pd.merge(
    df_2021,
    df_2023,
    on=merge_cols,
    suffixes=('_2021', '_2023')
)

# Clean confidence fields
merged_df['confidence_2021'] = merged_df['confidence_2021'].str.strip()
merged_df['confidence_2023'] = merged_df['confidence_2023'].str.strip()

# Filter unchanged Category 3
unchanged_category_3 = merged_df[
    (merged_df['confidence_2021'] == "3) Uncertain significance") &
    (merged_df['confidence_2023'] == "3) Uncertain significance")
]

print(f"Still Category 3 in both datasets: {unchanged_category_3.shape[0]}")
display(unchanged_category_3[["gene", "variant", "confidence_2021", "confidence_2023"]].head())


Still Category 3 in both datasets: 0


,gene,variant,confidence_2021,confidence_2023


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report


# Step 1: Filter Updated Variants
# Remove unchanged "3) Uncertain significance" cases
filtered_df = merged_df[
    ~((merged_df['confidence_2021'] == "3) Uncertain significance") &
      (merged_df['confidence_2023'] == "3) Uncertain significance"))
]
# now we are left with residues that have changed confidence levels
# Find entries where confidence values differ
updated_confidence = filtered_df[
    filtered_df['confidence_2021'] != filtered_df['confidence_2023']
].drop_duplicates()

# Print basic stats
print("Changes in confidence labels between two datasets:")
print(filtered_df['confidence_2021'].value_counts())
print(filtered_df['confidence_2023'].value_counts())
print('there are more resistant mutations in 2023')


Changes in confidence labels between two datasets:
confidence_2021
3) Uncertain significance     131
2) Assoc w R - Interim         73
1) Assoc w R                   72
5) Not assoc w R               35
4) Not assoc w R - Interim     22
Name: count, dtype: int64
confidence_2023
1) Assoc w R                  173
2) Assoc w R - Interim         81
3) Uncertain significance      39
4) Not assoc w R - Interim     20
5) Not assoc w R               20
Name: count, dtype: int64
there are more resistant mutations in 2023


In [ ]:

# Step 2: Get mutations reclassified from Category 3 → confident
# Must have been 3 in 2021, but something else in 2023
confidence_3_changed = updated_confidence[
    (updated_confidence['confidence_2021'] == "3) Uncertain significance") &
    (updated_confidence['confidence_2023'] != "3) Uncertain significance")
].copy()


print("New 2023 confidence label breakdown:")
confidence_3_dedup = confidence_3_changed.drop_duplicates(subset=['gene', 'one_letter_mutation'])
print(confidence_3_dedup['confidence_2023'].value_counts())
print(f"\nVariants reclassified from uncertain to confident from 2021 to 2023: {confidence_3_dedup.shape[0]}")


New 2023 confidence label breakdown:
confidence_2023
1) Assoc w R                  59
2) Assoc w R - Interim        27
4) Not assoc w R - Interim    12
5) Not assoc w R               6
Name: count, dtype: int64

Variants reclassified from uncertain to confident from 2021 to 2023: 104


In [ ]:
# Drop duplicates from both before merge
label_data_dedup = label_data.drop_duplicates(subset=['gene', 'one_letter_mutation'])
confidence_3_dedup = confidence_3_changed.drop_duplicates(subset=['gene', 'one_letter_mutation'])

# Merge again
reclassified_eval = pd.merge(
    label_data_dedup,
    confidence_3_dedup[['gene', 'one_letter_mutation', 'confidence_2023']],
    on=['gene', 'one_letter_mutation'],
    how='inner'
)
print(f"\nReclassified evaluation dataset shape: {reclassified_eval.shape}")


Reclassified evaluation dataset shape: (104, 16)


In [ ]:
# number of reclassified mutations
gene_counts = confidence_3_dedup['gene'].value_counts().reset_index()
gene_counts.columns = ['gene', 'reclassified_count']
print(gene_counts)

      gene  reclassified_count
0      gid                  22
1     pncA                  21
2     katG                  14
3     ethA                  13
4     rpoB                   8
5     embB                   8
6     inhA                   6
7   Rv0678                   4
8     gyrB                   4
9     gyrA                   2
10    rpsL                   1
11    rplC                   1


In [ ]:
gene_level_distribution = (confidence_3_dedup
    .groupby(['gene','confidence_2023'])
    .size()
    .reset_index(name='count')
    .sort_values(by= ['gene','count'], ascending=[True, False]))


print(gene_level_distribution)

      gene             confidence_2023  count
1   Rv0678      2) Assoc w R - Interim      3
0   Rv0678                1) Assoc w R      1
2     embB                1) Assoc w R      5
3     embB      2) Assoc w R - Interim      2
4     embB  4) Not assoc w R - Interim      1
6     ethA      2) Assoc w R - Interim     10
5     ethA                1) Assoc w R      3
7      gid                1) Assoc w R     16
8      gid      2) Assoc w R - Interim      3
9      gid  4) Not assoc w R - Interim      2
10     gid            5) Not assoc w R      1
11    gyrA      2) Assoc w R - Interim      1
12    gyrA  4) Not assoc w R - Interim      1
13    gyrB      2) Assoc w R - Interim      2
14    gyrB  4) Not assoc w R - Interim      1
15    gyrB            5) Not assoc w R      1
16    inhA                1) Assoc w R      4
17    inhA      2) Assoc w R - Interim      1
18    inhA            5) Not assoc w R      1
19    katG                1) Assoc w R      6
20    katG      2) Assoc w R - Int

In [ ]:
# ----------------------------
# Step 4: Predict using trained model

# Define the features used in training
numeric_columns = ['Proximity_to_R_Conferring']
# Fill missing values and prepare features
reclassified_eval[numeric_columns] = reclassified_eval[numeric_columns].fillna(0)

# Predict
reclassified_eval['model_prediction'] = trained_rf.predict(reclassified_eval[numeric_columns])

# Map binary labels as used in training
label_map = {
    "1) Assoc w R": 1,
    "2) Assoc w R - Interim": 1,
    "4) Not assoc w R - Interim": 0,
    "5) Not assoc w R": 0
}

# Map ground truth
reclassified_eval['true_label'] = reclassified_eval['confidence_2023'].map(label_map)


/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


In [ ]:
# Compare predictions
reclassified_eval['correct'] = reclassified_eval['model_prediction'] == reclassified_eval['true_label']
accuracy = reclassified_eval['correct'].mean() * 100

print(f"\n Accuracy of model on 104 reclassified variants: {accuracy:.2f}%")

# Optional breakdown
from sklearn.metrics import classification_report

print("\n Classification Report:")
print(classification_report(
    reclassified_eval['true_label'],
    reclassified_eval['model_prediction'],
    labels=[0, 1],
    target_names=['Susceptible', 'Resistant']
))



 Accuracy of model on 104 reclassified variants: 56.73%

 Classification Report:
              precision    recall  f1-score   support

 Susceptible       0.25      0.78      0.38        18
   Resistant       0.92      0.52      0.67        86

    accuracy                           0.57       104
   macro avg       0.59      0.65      0.53       104
weighted avg       0.80      0.57      0.62       104



In [ ]:
incorrect_preds = reclassified_eval[~reclassified_eval['correct']].copy()
print(f"Number of incorrect predictions: {incorrect_preds.shape[0]}")

Number of incorrect predictions: 45


In [ ]:
# Show relevant mutation-level info
print(incorrect_preds[['gene', 'one_letter_mutation', 'confidence_2023', 'true_label', 'model_prediction']])


       gene one_letter_mutation             confidence_2023  true_label  \
0      rpoB               I491L                1) Assoc w R           1   
1      rpoB               S493L                1) Assoc w R           1   
2      rpoB               I491M                1) Assoc w R           1   
3      rpoB               I491T                1) Assoc w R           1   
6      rpoB                S16C            5) Not assoc w R           0   
12     inhA               A239V      2) Assoc w R - Interim           1   
14     katG               S315I                1) Assoc w R           1   
15     katG               S315G                1) Assoc w R           1   
19     katG               P232A      2) Assoc w R - Interim           1   
20     katG               M257I      2) Assoc w R - Interim           1   
21     katG               G299S      2) Assoc w R - Interim           1   
22     katG               P232S      2) Assoc w R - Interim           1   
23     katG              

In [ ]:
print(incorrect_preds['gene'].value_counts())


gene
gid       8
katG      7
ethA      7
pncA      6
rpoB      5
embB      4
Rv0678    4
inhA      1
gyrA      1
rpsL      1
gyrB      1
Name: count, dtype: int64


In [ ]:
# Save predictions
reclassified_eval[['gene', 'one_letter_mutation', 'confidence_2023', 'true_label', 'model_prediction', 'correct']].to_csv(
    "../data/reclassified_104_variant_predictions.csv", index=False
)
